# FFNN Testing — `datasetml_2026.csv`
Binary classification: predict `placement_status` (Placed / Not Placed).

**Structure**
1. EDA & Preprocessing
2. Experiment A — Depth & Width
3. Experiment B — Activation Functions (hidden layers)
4. Experiment C — Learning Rate
5. Experiment D — Regularisation (None / L1 / L2)
6. Experiment E — Comparison with `sklearn` MLP

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))   # so we can import ffnn / activation / loss

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from ffnn import FFNN

SEED = 42
np.random.seed(SEED)
print("Imports OK")

## 1. EDA & Preprocessing

In [ ]:
df = pd.read_csv("../data/datasetml_2026.csv")
print("Shape:", df.shape)
display(df.head())
print("\nTarget distribution:")
print(df["placement_status"].value_counts())

In [ ]:
# ── Visualise target & numerical features ──────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

df["placement_status"].value_counts().plot(kind="bar", ax=axes[0], color=["steelblue","tomato"])
axes[0].set_title("Placement Status")
axes[0].set_xlabel(""); axes[0].set_ylabel("Count")

num_cols = ["cgpa","backlogs","internship_count","aptitude_score","communication_score","internship_quality_score"]
for ax, col in zip(axes[1:], num_cols):
    ax.hist(df[col], bins=40, color="steelblue", edgecolor="white")
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
# ── Preprocessing ──────────────────────────────────────────────────────────
data = df.copy()

# Encode target
data["label"] = (data["placement_status"] == "Placed").astype(int)

# Encode ordinal / low-cardinality categoricals with LabelEncoder
cat_cols = ["college_tier", "country", "university_ranking_band",
            "specialization", "industry"]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le

# Feature matrix and labels
feature_cols = ["cgpa","backlogs","college_tier","country","university_ranking_band",
                "internship_count","aptitude_score","communication_score",
                "specialization","industry","internship_quality_score"]
X_raw = data[feature_cols].values.astype(float)
y_raw = data["label"].values.reshape(-1, 1).astype(float)

# Train / val / test split  (70 / 15 / 15)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_raw, y_raw, test_size=0.15, random_state=SEED, stratify=y_raw)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765, random_state=SEED, stratify=y_trainval)

# Standardise numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

N_IN  = X_train.shape[1]
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

In [ ]:
# ── Helper utilities ───────────────────────────────────────────────────────
def evaluate(model, X, y_true, threshold=0.5):
    """Return accuracy for a binary model."""
    y_pred = model.predict(X)
    labels = (y_pred >= threshold).astype(int)
    return accuracy_score(y_true.astype(int), labels)

def plot_history(histories: dict, title: str):
    """Plot train / val loss curves for multiple runs."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for label, hist in histories.items():
        axes[0].plot(hist["train_loss"], label=label)
        if hist.get("val_loss"):
            axes[1].plot(hist["val_loss"], label=label)
    for ax, t in zip(axes, ["Train Loss", "Val Loss"]):
        ax.set_title(f"{title} — {t}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend()
    plt.tight_layout()
    plt.show()

BASE_PARAMS = dict(
    loss="binary_crossentropy",
    weight_init="xavier",
    init_params={"seed": SEED},
)
FIT_PARAMS = dict(
    X_val=X_val, y_val=y_val,
    batch_size=64, lr=0.01, epochs=50, verbose=0,
)
print("Helpers ready.")

## 2. Experiment A — Depth & Width
**Width variations** (depth fixed = 2 hidden layers)  
**Depth variations** (width fixed = 64 neurons per hidden layer)

In [ ]:
# ── A.1 Width variations (depth = 2 hidden layers) ─────────────────────────
width_configs = {
    "Narrow  [16,16]":  [N_IN, 16, 16, 1],
    "Medium  [64,64]":  [N_IN, 64, 64, 1],
    "Wide    [256,256]":[N_IN, 256, 256, 1],
}

width_histories = {}
width_accs = {}
for name, sizes in width_configs.items():
    model = FFNN(sizes, ["relu","relu","sigmoid"], **BASE_PARAMS)
    hist  = model.fit(X_train, y_train, **FIT_PARAMS)
    width_histories[name] = hist
    width_accs[name] = evaluate(model, X_test, y_test)
    print(f"{name}  →  test acc = {width_accs[name]:.4f}")

plot_history(width_histories, "Width Variation (depth=2)")

In [ ]:
# ── A.2 Depth variations (width = 64 per hidden layer) ─────────────────────
depth_configs = {
    "Shallow  (1 hidden)": [N_IN, 64, 1],
    "Medium   (3 hidden)": [N_IN, 64, 64, 64, 1],
    "Deep     (5 hidden)": [N_IN, 64, 64, 64, 64, 64, 1],
}

depth_histories = {}
depth_accs = {}
for name, sizes in depth_configs.items():
    acts = ["relu"] * (len(sizes) - 2) + ["sigmoid"]
    model = FFNN(sizes, acts, **BASE_PARAMS)
    hist  = model.fit(X_train, y_train, **FIT_PARAMS)
    depth_histories[name] = hist
    depth_accs[name] = evaluate(model, X_test, y_test)
    print(f"{name}  →  test acc = {depth_accs[name]:.4f}")

plot_history(depth_histories, "Depth Variation (width=64)")

In [ ]:
# ── A summary table ─────────────────────────────────────────────────────────
rows = [{"Config": k, "Type": "Width", "Test Acc": v} for k, v in width_accs.items()]
rows += [{"Config": k, "Type": "Depth", "Test Acc": v} for k, v in depth_accs.items()]
pd.DataFrame(rows).sort_values("Test Acc", ascending=False).reset_index(drop=True)

## 3. Experiment B — Activation Functions (Hidden Layers)
Base architecture: `[N_IN → 128 → 64 → 32 → 1]`.  
Layer 2 (index 1, i.e. the 64-neuron layer) is the test layer — its activation is varied.  
Candidates: `linear`, `relu`, `sigmoid`, `tanh` (softmax excluded for hidden layers).

In [ ]:
# ── B Activation function experiment ───────────────────────────────────────
# Base arch: [N_IN, 128, 64, 32, 1]  — test layer index 1 (acts[1])
test_activations = ["linear", "relu", "sigmoid", "tanh"]

act_histories = {}
act_accs      = {}
act_models    = {}

for act in test_activations:
    acts  = ["relu", act, "relu", "sigmoid"]   # layer 1 varied
    model = FFNN([N_IN, 128, 64, 32, 1], acts, **BASE_PARAMS)
    hist  = model.fit(X_train, y_train, **FIT_PARAMS)
    act_histories[act] = hist
    act_accs[act]      = evaluate(model, X_test, y_test)
    act_models[act]    = model
    print(f"act={act:<8}  →  test acc = {act_accs[act]:.4f}")

plot_history(act_histories, "Activation Function on Layer 2")

In [ ]:
# ── B  Weight & gradient distributions for all models ──────────────────────
for act, model in act_models.items():
    print(f"\n── Activation = {act} ──")
    model.plot_weight_distribution(layers=[0, 1, 2, 3])
    model.plot_gradient_distribution(layers=[0, 1, 2, 3])

# Summary table
pd.DataFrame({"Activation": list(act_accs.keys()),
              "Test Acc":   list(act_accs.values())}).sort_values("Test Acc", ascending=False).reset_index(drop=True)

## 4. Experiment C — Learning Rate
Three learning-rate values: `0.001`, `0.01`, `0.1`.  
Fixed architecture `[N_IN → 128 → 64 → 1]`, activations `relu / relu / sigmoid`.

In [ ]:
# ── C Learning rate experiment ──────────────────────────────────────────────
lr_values = [0.001, 0.01, 0.1]
arch      = [N_IN, 128, 64, 1]
acts      = ["relu", "relu", "sigmoid"]

lr_histories = {}
lr_accs      = {}
lr_models    = {}

for lr in lr_values:
    model = FFNN(arch, acts, **BASE_PARAMS)
    hist  = model.fit(X_train, y_train, **{**FIT_PARAMS, "lr": lr})
    label = f"lr={lr}"
    lr_histories[label] = hist
    lr_accs[label]      = evaluate(model, X_test, y_test)
    lr_models[label]    = model
    print(f"{label}  →  test acc = {lr_accs[label]:.4f}")

plot_history(lr_histories, "Learning Rate Comparison")

In [ ]:
# ── C  Weight & gradient distributions ─────────────────────────────────────
for label, model in lr_models.items():
    print(f"\n── {label} ──")
    model.plot_weight_distribution(layers=[0, 1, 2])
    model.plot_gradient_distribution(layers=[0, 1, 2])

pd.DataFrame({"LR": list(lr_accs.keys()),
              "Test Acc": list(lr_accs.values())}).sort_values("Test Acc", ascending=False).reset_index(drop=True)

## 5. Experiment D — Regularisation
Three runs: no regularisation, L1 (λ=0.001), L2 (λ=0.001).  
Same architecture `[N_IN → 128 → 64 → 1]`, `relu / relu / sigmoid`, `lr=0.01`.

In [ ]:
# ── D Regularisation experiment ─────────────────────────────────────────────
LAMBDA = 0.001
reg_configs = [
    ("No Regularisation", dict(l1=0.0,    l2=0.0   )),
    ("L1  λ=0.001",       dict(l1=LAMBDA, l2=0.0   )),
    ("L2  λ=0.001",       dict(l1=0.0,    l2=LAMBDA)),
]

reg_histories = {}
reg_accs      = {}
reg_models    = {}

for name, reg_kwargs in reg_configs:
    model = FFNN(
        [N_IN, 128, 64, 1], ["relu","relu","sigmoid"],
        loss="binary_crossentropy",
        weight_init="xavier", init_params={"seed": SEED},
        **reg_kwargs,
    )
    hist = model.fit(X_train, y_train, **FIT_PARAMS)
    reg_histories[name] = hist
    reg_accs[name]      = evaluate(model, X_test, y_test)
    reg_models[name]    = model
    print(f"{name}  →  test acc = {reg_accs[name]:.4f}")

plot_history(reg_histories, "Regularisation Comparison")

In [ ]:
# ── D  Weight & gradient distributions ─────────────────────────────────────
for name, model in reg_models.items():
    print(f"\n── {name} ──")
    model.plot_weight_distribution(layers=[0, 1, 2])
    model.plot_gradient_distribution(layers=[0, 1, 2])

pd.DataFrame({"Regularisation": list(reg_accs.keys()),
              "Test Acc": list(reg_accs.values())}).reset_index(drop=True)

## 6. Experiment E — Comparison with `sklearn` MLP
Train our FFNN and `sklearn.neural_network.MLPClassifier` with **identical** hyperparameters and compare test accuracy & classification report.

In [ ]:
from sklearn.neural_network import MLPClassifier

# ── Shared hyperparameters ──────────────────────────────────────────────────
HIDDEN  = (128, 64)   # two hidden layers: 128 → 64
LR_CMP  = 0.01
EPOCHS  = 50
BATCH   = 64

# ── Our FFNN ────────────────────────────────────────────────────────────────
our_model = FFNN(
    [N_IN, 128, 64, 1], ["relu", "relu", "sigmoid"],
    loss="binary_crossentropy",
    weight_init="xavier", init_params={"seed": SEED},
)
our_model.fit(X_train, y_train, X_val=X_val, y_val=y_val,
              batch_size=BATCH, lr=LR_CMP, epochs=EPOCHS, verbose=0)

our_preds  = (our_model.predict(X_test) >= 0.5).astype(int).flatten()
our_acc    = accuracy_score(y_test.astype(int).flatten(), our_preds)

# ── sklearn MLPClassifier ───────────────────────────────────────────────────
sklearn_model = MLPClassifier(
    hidden_layer_sizes=HIDDEN,
    activation="relu",
    solver="sgd",
    learning_rate_init=LR_CMP,
    batch_size=BATCH,
    max_iter=EPOCHS,
    random_state=SEED,
    early_stopping=False,
)
sklearn_model.fit(X_train, y_train.flatten().astype(int))
sk_preds = sklearn_model.predict(X_test)
sk_acc   = accuracy_score(y_test.astype(int).flatten(), sk_preds)

print(f"Our FFNN  accuracy : {our_acc:.4f}")
print(f"sklearn MLP accuracy: {sk_acc:.4f}")

In [ ]:
# ── Detailed classification reports ─────────────────────────────────────────
target_names = ["Not Placed", "Placed"]
y_test_int   = y_test.astype(int).flatten()

print("═" * 45)
print("Our FFNN")
print("═" * 45)
print(classification_report(y_test_int, our_preds, target_names=target_names))

print("═" * 45)
print("sklearn MLPClassifier")
print("═" * 45)
print(classification_report(y_test_int, sk_preds, target_names=target_names))

# ── Bar chart comparison ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Our FFNN", "sklearn MLP"], [our_acc, sk_acc],
       color=["steelblue", "tomato"], width=0.4)
ax.set_ylim(0, 1)
ax.set_ylabel("Test Accuracy")
ax.set_title("Our FFNN vs sklearn MLP — Test Accuracy")
for i, v in enumerate([our_acc, sk_acc]):
    ax.text(i, v + 0.01, f"{v:.4f}", ha="center", fontsize=11)
plt.tight_layout()
plt.show()

## Extra — Save & Load Test

In [ ]:
# Save the best model (our FFNN from experiment E) and reload it
our_model.save("model_best.pkl")
loaded = FFNN.load("model_best.pkl")

# Verify predictions match
orig_preds   = (our_model.predict(X_test)  >= 0.5).astype(int)
loaded_preds = (loaded.predict(X_test)     >= 0.5).astype(int)
assert np.array_equal(orig_preds, loaded_preds), "Save/load mismatch!"
print("Save & Load verified  ✓")
print("Loaded model repr:")
print(loaded)